In [1]:
!pip install langchain langchain_core langchain_openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.6/987.6 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.7/371.7 kB 12.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.5/328.5 kB 16.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.9 MB/s eta 0:00:00


In [2]:
import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass.getpass()

··········


In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field

from langchain_core.messages import AIMessage
from langchain_core.runnables import Runnable, RunnablePassthrough
from langchain_core.tools import tool

from langchain_openai import ChatOpenAI

In [4]:
customer_birthdays = [
    {"customer_id": 1, "name": "Alice", "birthday": "12/31/1985"},  # MM/DD/YYYY format
    {"customer_id": 2, "name": "Bob", "birthday": "1986-01-01"},    # YYYY-MM-DD format
    {"customer_id": 3, "name": "Charlie", "birthday": "03/15/85"},  # MM/DD/YY format
    {"customer_id": 4, "name": "Diana", "birthday": "85/03/15"},    # YY/MM/DD format (incorrect)
    {"customer_id": 5, "name": "Eve", "birthday": "01-45-1990"},    # Invalid date (month out of range)
    {"customer_id": 6, "name": "Frank", "birthday": "07/25/2025"},  # Date in the future
    {"customer_id": 7, "name": "Grace", "birthday": "1990.07.25"},  # YYYY.MM.DD format
    {"customer_id": 8, "name": "Heidi", "birthday": "25/07/1990"},  # DD/MM/YYYY format
    {"customer_id": 9, "name": "Ivan", "birthday": "1990/13/25"},   # Invalid date (month out of range)
    {"customer_id": 10, "name": "Judy", "birthday": "11-31-1990"},  # Invalid date (day out of range)
    {"customer_id": 11, "name": "Kevin", "birthday": "19900101"},   # YYYYMMDD format
    {"customer_id": 12, "name": "Laura", "birthday": "90-01-01"},   # YY-MM-DD format
    {"customer_id": 13, "name": "Mallory", "birthday": "July 25, 1990"},  # Month Day, Year format
    {"customer_id": 14, "name": "Niaj", "birthday": "1990 July 25"}, # Year Month Day format
    {"customer_id": 15, "name": "Oscar", "birthday": "Feb 30 1990"}, # Invalid date (February 30th)
    {"customer_id": 16, "name": "Peggy", "birthday": "00/00/0000"},  # Placeholder date (invalid)
    {"customer_id": 17, "name": "Quinn", "birthday": "1985/02/30"},  # Invalid date (February 30th)
    {"customer_id": 18, "name": "Ruth", "birthday": ""},             # Missing date
    {"customer_id": 19, "name": "Sybil", "birthday": "August 15, 1985"},  # Month Day, Year format
    {"customer_id": 20, "name": "Trudy", "birthday": "1985-08-15"},  # YYYY-MM-DD format (correct)
]

In [5]:
class CustomerRecord(BaseModel):
    customer_id: int = Field(alias="Customer id")
    name: str = Field(alias="Customer name")
    birthday: str = Field(alias="Customer birthday")
    action: str = Field(alias="Action to take. Possible values are 'update record' or 'contact customer'")

In [6]:
llm = ChatOpenAI(temperature=0)

In [7]:
birthday_prompt = """Customer birthday record:
{record}

The customer's birthday in the record above might not be in YYYY-MM-DD format. Please review the record and correct any errors.

If the date is missing or invalid, please indicate if the customer should be contacted for a new birthday. Otherwise, indicate if the customer's birthday should be updated with the new birthday.
"""

prompt = PromptTemplate(
    template = birthday_prompt,
    input_variables=["record"]
)

In [8]:
structured_llm = llm.with_structured_output(CustomerRecord)

In [9]:
structured_chain = prompt | structured_llm

In [10]:
customer_birthdays[0]

{'customer_id': 1, 'name': 'Alice', 'birthday': '12/31/1985'}

In [11]:
structured_chain.invoke({"record": customer_birthdays[0]})

CustomerRecord(customer_id=1, name='Alice', birthday='1985-12-31', action='update record')

In [12]:
customer_birthdays[15]

{'customer_id': 16, 'name': 'Peggy', 'birthday': '00/00/0000'}

In [13]:
structured_chain.invoke({"record": customer_birthdays[15]})

CustomerRecord(customer_id=16, name='Peggy', birthday='00/00/0000', action='contact customer')

In [14]:
def update_record(updated_record: dict) -> str:
  """Update the customer record in the database"""
  for i in range(len(customer_birthdays)):
    if customer_birthdays[i]["customer_id"] == updated_record.customer_id:
      customer_birthdays[i]["birthday"] = updated_record.birthday
      print(f"Updated birthday for customer {updated_record.customer_id}")

def contact_customer(customer_record: int) -> str:
  """Contact the customer to get a new birthday"""
  print(f"Contacting customer {customer_record.name}...")

In [15]:
def human_approval(corrected_record: CustomerRecord) -> str:
    input_msg = ("Do you approve of the '{0}' action for the corrected customer record?\n\nName: {1}\nBirthday: {2}\n\n".format(corrected_record.action,
                                                                                                                                 corrected_record.name,
                                                                                                                                 corrected_record.birthday))
    resp = input(input_msg)
    if resp.lower() == "yes":
        if corrected_record.action == "update record":
            update_record(corrected_record)
        else:
            contact_customer(corrected_record)
    else:
        return f"Rejected '{corrected_record.action}' action for customer {corrected_record.customer_id}"

In [16]:
structured_chain = prompt | structured_llm | human_approval

In [17]:
structured_chain.invoke({"record": customer_birthdays[0]})

Do you approve of the 'update record' action for the corrected customer record?

Name: Alice
Birthday: 1985-12-31

yes
Updated birthday for customer 1


In [18]:
customer_birthdays[0]

{'customer_id': 1, 'name': 'Alice', 'birthday': '1985-12-31'}

In [19]:
structured_chain.invoke({"record": customer_birthdays[15]})

Do you approve of the 'contact customer' action for the corrected customer record?

Name: Peggy
Birthday: 00/00/0000

no


"Rejected 'contact customer' action for customer 16"